# 07 · Evaluate detectors on your own data without labels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/07_evaluate_without_labels.ipynb)

**Runtime:** CPU is fine for everything except the Clef-flash section. That section needs a GPU because Clef-flash is a 9.4-billion-parameter model whose bf16 weights take about 18.8 GB. An L4 (24 GB with native bf16) is recommended. A T4 (16 GB) can try 8-bit weights because the bf16 weights do not fit in its memory. That 8-bit path has not been tested yet and its probabilities may differ from an L4 run. TPU runtimes are not supported because the model's loader targets CUDA and 8-bit loading through bitsandbytes does not run on TPUs. Without a GPU that section is skipped and the rest of the notebook still runs.

← [06_search_on_sanitized_data](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/06_search_on_sanitized_data.ipynb) · [series index](https://github.com/jstjoe/local-privacy/blob/main/notebooks/README.md)

Notebooks 00 to 06 score detectors against datasets that ship gold spans. Your own support tickets or contracts or spreadsheets have no gold spans. This notebook runs detectors on your files and then estimates precision and recall in three ways that you can mix:

1. **Silver labels (option A).** One or more LLMs annotate your data. The normal report then scores the detectors against those annotations as if they were gold.
2. **LLM review (option B).** An LLM judges every span the detectors found and lists the personal data that every detector missed.
3. **Classifier review (option C).** Cloudflare's Clef-flash runs locally and answers yes/no questions about each span and each sentence.

None of these is ground truth. Every estimate states its bias next to the number. The last section reruns the chosen methods on a sample with real gold labels so you can see how far each estimate lands from the true score before you trust it on your data.

**Work on a copy when you run this on your own files.** Jupyter saves every output into the notebook file, and the outputs here quote the personal data found in your files. This notebook is tracked by git in the repository, so running it in place would put that data into a file that `git add -A` commits. Save a copy outside the repository first or clear all outputs before you commit. The setup cell prints a warning when it finds that it runs from the tracked file.

In [ ]:
# Setup — run once per session. On Colab this clones and installs the harness
# (a few minutes the first time). Locally, after `uv sync --all-packages --all-extras`, it does nothing.
HARNESS_REPO = "https://github.com/jstjoe/local-privacy.git"
HARNESS_REF = "main"   # branch or tag — e.g. a PR branch to try it before merging
OPF_REPO = "https://github.com/openai/privacy-filter.git"
OPF_REF = "main"
USE_DRIVE = False      # True: keep results in Google Drive so the other notebooks reuse them

import importlib.util, os, shutil, subprocess, sys

if importlib.util.find_spec("opf_eval") is None:
    home = "/content" if "google.colab" in sys.modules else os.path.expanduser("~")
    for repo, ref, dest in [(OPF_REPO, OPF_REF, f"{home}/privacy-filter"),
                            (HARNESS_REPO, HARNESS_REF, f"{home}/local-privacy")]:
        if not os.path.exists(dest):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", ref, repo, dest], check=True)
    # File parsing and the LLM clients always. The Clef-flash libraries only on a
    # CUDA GPU because the model cannot run without one.
    extras = "notebooks,files,llm" + (",clef" if shutil.which("nvidia-smi") else "")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{home}/privacy-filter",
                    f"{home}/local-privacy/eval[{extras}]", f"{home}/local-privacy/api"], check=True)
    # A running kernel doesn't always see freshly pip-installed packages;
    # putting the sources on sys.path makes the imports below work either way.
    sys.path[:0] = [f"{home}/privacy-filter", f"{home}/local-privacy/eval/src", f"{home}/local-privacy/api/src"]

from opf_eval import nb
ws = nb.setup(drive=USE_DRIVE)
nb.warn_if_tracked("07_evaluate_without_labels.ipynb")   # outputs quote your data, so work on a copy

## 1 · Setup and data policy

The switch `PII_BENCH_ALLOW_REMOTE` decides whether hosted LLMs may see your data. Every hosted call in this notebook checks it first, and `opf_eval.llm.make_client` refuses a hosted provider when it is off. Each client prints who operates it before its first call. The next cell reads the switch in this order:

1. `ALLOW_REMOTE = False` or `ALLOW_REMOTE = True` in the cell sets the switch for this kernel.
2. `ALLOW_REMOTE = None` uses the value of `PII_BENCH_ALLOW_REMOTE` from your shell or your `.env` file or Colab **Secrets**. The value `0` forbids hosted calls. Going back to `None` after `True` or `False` undoes that override.
3. When the variable is not set anywhere, hosted calls are allowed for now.

**Data exposure policy for now.** During internal development it is acceptable for Colab and OpenAI and Anthropic to see the data this notebook processes. Hosted calls are therefore allowed by default for now. This must be tightened before the notebook is shared or rolled out more widely. That is the reason the LLM clients support Google Vertex AI now and will support Amazon Bedrock in a later phase. Both run the models inside the team's own cloud account and region under that account's data terms. Before a wider rollout the default flips to forbidding hosted calls and this notebook will document which providers are approved.

The cell also loads the credentials. On Colab add them under **Secrets** (the key icon in the left sidebar). Locally set them in your shell or in a `.env` file. Each provider needs one of them:

1. `ANTHROPIC_API_KEY` is for Claude through the Anthropic API.
2. `OPENAI_API_KEY` is for GPT models through the OpenAI API.
3. `GOOGLE_CLOUD_PROJECT` is for Claude or gpt-oss on Vertex AI in your own GCP project. Vertex AI also needs Google application default credentials, which the notebook asks for only when you pick a Vertex provider.
4. `OPENAI_COMPATIBLE_API_KEY` is for an `openai_compatible` server such as vLLM or Ollama. You need it only when that server asks for a key. Name such a server as `("openai_compatible", model, base_url)` in `LABELERS` or `REVIEWER`, for example `("openai_compatible", "qwen3:8b", "http://localhost:11434/v1")`. A server on this machine keeps the text local, so it runs even when hosted calls are switched off.

A rerun costs nothing and gives the same numbers because LLM answers are cached on disk in the workspace's `llm_cache` folder. Keep that cache as private as your files because it holds the PII values that the models quoted back.

Before each hosted step the notebook prints the number of calls and the input and output tokens and a rough time. Multiply each token count by your model's price per million tokens to get the cost.

**Dry run.** When the environment variable `PII_BENCH_NOTEBOOK_STUB` is `1` the notebook swaps every LLM for a local stub and Clef-flash for a stub backend. It then runs end to end with no keys and no GPU. It is not fully offline because the detectors still run for real. The first dry run therefore installs the spaCy model `en_core_web_lg` for Presidio and downloads the GLiNER weights from Hugging Face. Set `PII_BENCH_NOTEBOOK_DETECTORS=presidio` to skip the GLiNER download. The numbers from a dry run mean nothing because the stubs are toy pattern matchers.

In [ ]:
# === EDIT ME ===
# None keeps PII_BENCH_ALLOW_REMOTE from the shell or .env or Colab Secrets (allowed while unset).
# False forbids hosted LLMs in this kernel and True allows them.
ALLOW_REMOTE = None
# ================
# PII_BENCH_NOTEBOOK_STUB=1 runs the notebook with stub LLMs and a stub Clef-flash. The detectors still run for real.
USE_STUB = os.environ.get("PII_BENCH_NOTEBOOK_STUB") == "1"

import math
from collections import Counter
from pathlib import Path

from opf_eval import documents, fixtures, llm, report, review, scoring, silver
from opf_eval.io import read_jsonl, read_meta
from opf_eval.review import classifier, clef
from opf_eval.review import llm as review_llm

# The credential each hosted provider needs before a client is worth building.
NEEDS = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY",
         "anthropic_vertex": "GOOGLE_CLOUD_PROJECT", "openai_vertex": "GOOGLE_CLOUD_PROJECT"}

nb.load_secrets(nb.ALLOW_REMOTE_ENV)   # a value set in the shell or .env or Colab Secrets
nb.set_allow_remote(ALLOW_REMOTE)      # None undoes an override from an earlier run of this cell
found = nb.load_secrets(*nb.LLM_SECRETS)   # OPENAI_COMPATIBLE_API_KEY is optional
LLM_CACHE = ws.root / "llm_cache"
print(f"hosted LLMs may see the data: {nb.allow_remote()}")
print("credentials: " + ", ".join(f"{k} {'found' if v else 'missing'}" for k, v in found.items()))
print(f"LLM cache:   {LLM_CACHE}")
if USE_STUB:
    print("DRY RUN: every LLM is a local stub and Clef-flash is a stub backend. The numbers mean nothing.")

SECONDS_PER_CALL = 20   # a rough latency for one uncached call on a 4,000-character chunk


def call_estimate(n_calls, input_tokens, output_tokens, workers=4):
    """A one-line cost and time note to print before any hosted call."""
    minutes = math.ceil(n_calls / workers) * SECONDS_PER_CALL / 60
    took = "under a minute" if minutes < 1 else f"about {minutes:.0f} minutes"
    return (f"{n_calls} calls with roughly {input_tokens:,} input tokens and up to {output_tokens:,} output "
            f"tokens. They take {took} at {workers} calls at a time. Multiply each token count by your "
            "model's price per million tokens to get the cost. Cached answers cost nothing and take no time.")


def llm_client(provider, model, base_url=None, *, stub_responder):
    """Return `(client, None)` for one LLM, or `(None, reason)` when it cannot run here.

    `base_url` is the address of an `openai_compatible` server and is ignored by
    the other providers. In a dry run the client is a local stub that answers
    with `stub_responder`.
    """
    if USE_STUB:
        return llm.make_client("stub", model=model, responder=stub_responder), None
    url = {"base_url": base_url} if base_url else {}
    try:
        info = llm.provider_info(provider, **url)
    except ValueError as e:
        return None, str(e)
    if info.remote and not nb.allow_remote():
        return None, f"{provider} would send the text to {info.operator} and hosted LLMs are switched off"
    key = NEEDS.get(provider)
    if key and not os.environ.get(key):
        return None, f"{key} is not set"
    # nb.gcloud_ready asks again on every run until the sign-in works once.
    if provider.endswith("_vertex") and not nb.gcloud_ready():
        return None, "Vertex AI needs Google application default credentials"
    try:
        return llm.make_client(provider, model, cache_dir=LLM_CACHE, **url), None
    except (llm.LLMError, ImportError, ValueError) as e:
        return None, f"{type(e).__name__}: {e}"

### Dry-run stand-ins

The next cell defines the toy stand-ins that replace the models in a dry run. They find a few kinds of value with regular expressions. Nothing in the cell calls a model, and the rest of the notebook uses the stand-ins only when `USE_STUB` is `True`.

In [ ]:
import json, re

from opf_eval import taxonomy

# Toy patterns for the dry run, most specific first. Real runs never use them.
TOY_PATTERNS = {
    "CREDIT_CARD": r"\b(?:\d{4}[ -]){3}\d{4}\b",
    "EMAIL": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    "DATE_OF_BIRTH": r"\b\d{1,2}/\d{1,2}/\d{4}\b",
    "PHONE": r"(?<![\w-])\+?\(?\d[\d ()-]{7,}\d\b",
}
TOY_LOOKS_LIKE_PII = re.compile(r"[\d@]|[A-Z]\w+ [A-Z]\w+")   # a digit or an @ or two capitalised words


def toy_find(text, kinds):
    """Every match of the toy patterns for `kinds` as silver-label items."""
    return [{"value": m.group(), "label": kind, "context": m.group()}
            for kind in kinds for m in re.finditer(TOY_PATTERNS[kind], text)]


def toy_labeler(kinds):
    """A stub silver labeler that finds only the values of `kinds`."""
    def respond(system, user, schema):
        allowed = set(schema["properties"]["entities"]["items"]["properties"]["label"]["enum"])
        text = user.split("<text>\n", 1)[-1]
        return {"entities": toy_find(text, [k for k in kinds if k in allowed])}
    return respond


CANDIDATE = re.compile(r'^\[(\d+)\] .*? ("(?:[^"\\]|\\.)*") at \d+-\d+:', re.M)


def toy_reviewer(system, user, schema):
    """A stub LLM reviewer built on `review.llm.dry_run_responder`.

    It rejects candidates with no digit and no @ that are not two capitalised
    words. It reports every toy-pattern match as a miss, and the review drops
    the misses that a candidate covers.
    """
    answer = review_llm.dry_run_responder(system, user, schema)
    texts = {int(i): json.loads(s) for i, s in CANDIDATE.findall(user)}
    for v in answer["verdicts"]:
        if not TOY_LOOKS_LIKE_PII.search(texts.get(v["index"], "")):
            v["verdict"], v["note"] = "not_pii", "dry run"
    allowed = set(schema["properties"]["missed"]["items"]["properties"]["label"]["enum"])
    body = user.split("<<<\n", 1)[-1].rsplit("\n>>>", 1)[0]
    answer["missed"] = toy_find(body, [k for k in TOY_PATTERNS if k in allowed])
    return answer


def toy_decisions(request):
    """A stub Clef-flash: the reviewer's rule of thumb expressed as probabilities."""
    questions, state = request["questions"], request["state"]
    if "is_pii" in questions:
        p = 0.9 if TOY_LOOKS_LIKE_PII.search(state["span"]) else 0.2
        return {
            "is_pii": classifier.noul_answer(p),
            "label_ok": classifier.noul_answer(0.8 if p > 0.5 else 0.3),
            "label": classifier.choice_answer({state["proposed_label"]: p, classifier.NOT_PII: 1 - p}),
        }
    left = next((k for k, pat in TOY_PATTERNS.items() if re.search(pat, state["text"])), None)
    kind = taxonomy.parent(left) if left else classifier.NONE
    return {
        "residual": classifier.noul_answer(0.85 if left else 0.1),
        "residual_type": classifier.choice_answer({kind: 0.9, **({classifier.NONE: 0.1} if left else {})}),
    }

## 2 · Bring your data

The notebook reads every supported file in one folder. By default that is the workspace's `inputs` folder. When the folder is empty and `PASTED_TEXTS` holds no text the next cell writes a small sample corpus into it so the notebook works with no uploads. The sample holds one e-mail and one CSV file and one text file with fake personal data. Delete those three files once you add your own.

You can bring your own data in four ways:

1. Copy files into the `inputs` folder on your machine. The cell prints where it is.
2. On Colab uncomment `nb.upload_files()` to pick files from your computer. They land in the same folder.
3. On Colab point `INPUT_DIR` at a folder in your Google Drive with `nb.drive_folder(...)`.
4. Paste strings into `PASTED_TEXTS`.

Every file becomes plain text plus a map from each character back to its page or cell or line or e-mail header. Detectors only see the text. The map lets the notebook show a detected value as `People!B2` or `page 3` instead of a character offset. The parser reads these file types:

1. PDF through its text layer. Scanned pages with no text layer are reported as needing OCR and skipped.
2. Word `.docx` with its tables and headers and footers.
3. Excel `.xlsx` and `.xlsm` with one segment per cell.
4. CSV and TSV with one segment per cell.
5. Plain text and Markdown as written.
6. HTML through its visible text.
7. E-mail `.eml` with its headers and body and supported attachments.

PDF and Word and Excel need the `files` extra, which the setup cell installs on Colab.

In [ ]:
# === EDIT ME ===
INPUT_DIR = nb.inputs_dir()
# INPUT_DIR = nb.drive_folder("pii-inputs")   # Colab: the folder "pii-inputs" in My Drive
# nb.upload_files()                            # Colab: upload files into nb.inputs_dir()
PASTED_TEXTS = []   # e.g. ["Please call Ana Silva on +351 912 345 678 about her refund."]
# ================

# A tiny synthetic corpus so the notebook runs with no uploads. Every value in it is fake.
SAMPLE_FILES = {
    "support_ticket.eml": """\
From: Maria Gonzalez <maria.gonzalez@example.com>
To: Support Team <support@example.org>
Subject: Refund for order 48213
Date: Mon, 6 Oct 2025 09:14:00 +0000
Content-Type: text/plain; charset=utf-8

Hello,

My name is Maria Gonzalez and I was charged twice for order 48213.
The card 4111 1111 1111 1111 was billed on 2 October.
Please call me back on +1 415 555 0142 or write to maria.gonzalez@example.com.
My billing address is 1200 Market Street, San Francisco, CA 94102.

Thanks,
Maria
""",
    "customers.csv": """\
name,email,phone,city,notes
James Carter,james.carter@example.net,(212) 555-0187,New York,Prefers email contact
Priya Natarajan,priya.n@example.com,+44 20 7946 0958,London,IBAN GB82 WEST 1234 5698 7654 32
Lukas Weber,l.weber@example.de,+49 30 901820,Berlin,Born 14 March 1986
""",
    "call_notes.txt": """\
Call notes, 3 October

Spoke with Daniel Okafor (customer ID 99-1043) about his account.
He confirmed his date of birth as 22/07/1979 and his SSN 219-09-9999.
Follow up by email at dokafor@example.org or on 07700 900123.
The meeting with Sarah Lin moved to Friday at the Seattle office.
""",
}
if (INPUT_DIR == nb.inputs_dir() and not any(t.strip() for t in PASTED_TEXTS)
        and not any(p.is_file() and not p.name.startswith(".") for p in INPUT_DIR.rglob("*"))):
    for name, content in SAMPLE_FILES.items():
        (INPUT_DIR / name).write_text(content)
    print("the sample corpus was written into the empty folder because there was no other data")
print(f"reading files from {INPUT_DIR}")

In [ ]:
docs, skipped = documents.parse_many(INPUT_DIR)
docs += [documents.from_text(t, id=f"pasted-{i}", source=f"<pasted text {i}>")
         for i, t in enumerate(PASTED_TEXTS, 1) if t.strip()]
if not docs:
    raise RuntimeError(f"no documents to evaluate: add files to {INPUT_DIR} or text to PASTED_TEXTS")


def doc_name(doc):
    """The document's path relative to the input folder, or its pasted-text id."""
    return doc.id.split(":", 1)[-1]


print(f"parsed {len(docs)} document(s)")
for s in skipped:
    print(f"skipped {s['path']}: {s['reason']}")
if not skipped:
    print("no file was skipped")
for doc in docs:
    for w in doc.warnings:
        print(f"warning in {doc_name(doc)}: {w}")
nb.show_table([{"document": doc_name(d), "kind": d.kind, "characters": len(d.text),
                "segments": len(d.segments), "warnings": len(d.warnings)} for d in docs])

### Preview the segments

Each document's first segments show where its text came from. Check that the parser read your files the way you expect before you spend time on detectors. The previews in this notebook show the first `SHOW_DOCS` documents so a large folder does not flood the page.

In [ ]:
# === EDIT ME ===
SHOW_DOCS = 5          # documents shown in each preview of this notebook
PREVIEW_SEGMENTS = 6   # segments shown per document
# ================


def shown_docs(items):
    """The first SHOW_DOCS items, after saying how many more there are."""
    items = list(items)
    if len(items) > SHOW_DOCS:
        print(f"showing {SHOW_DOCS} of {len(items)} documents. Raise SHOW_DOCS to see more.")
    return items[:SHOW_DOCS]


for doc in shown_docs(docs):
    nb.show_md(f"**{doc_name(doc)}** · {doc.kind} · {len(doc.segments)} segments")
    nb.show_table([{"where": nb.format_where(seg.where), "text": doc.segment_text(seg)[:100]}
                   for seg in doc.segments[:PREVIEW_SEGMENTS]])

### Write the fixtures

The documents become ordinary fixture records with no gold spans. Long documents are split into chunks of at most 4,000 characters at segment boundaries. Each chunk remembers its document and its offset so every detected span can be put back on the whole document. The sidecar file records that the fixtures are unlabeled, which makes the normal report refuse to print precision and recall for them.

In [ ]:
FIXTURES = ws.data / "custom" / "your_data.jsonl"
FIXTURES.parent.mkdir(parents=True, exist_ok=True)
fx = fixtures.from_documents(docs, FIXTURES)
print(fixtures.validate_fixtures(fx).summary())

## 3 · Run detectors

`nb.ensure_run` runs each detector over every chunk and saves the predictions in a run directory named after the fixtures file. It runs the detectors again whenever the fixtures file changes. Notebooks 01 to 06 keep the dataset sample they share because the session here is not saved.

Presidio and GLiNER are the defaults because they need no credentials and are quick on a CPU. Any detector from [02](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/02_run_detectors.ipynb) works here. Detectors are asked for every canonical label because your data has no annotated label set.

Skyflow needs `SKYFLOW_VAULT_URL` and `SKYFLOW_VAULT_ID` and `SKYFLOW_BEARER_TOKEN`. Add them under Colab Secrets (the key icon) or to `.env` and the cell below loads them when `DETECTORS` lists `skyflow` or `skyflow_full`. The cell stops with the names of any that are missing.

In [ ]:
# === EDIT ME ===
DETECTORS = ["presidio", "gliner"]
# ================
# A test run can override the list, e.g. PII_BENCH_NOTEBOOK_DETECTORS=presidio.
if os.environ.get("PII_BENCH_NOTEBOOK_DETECTORS"):
    DETECTORS = [d.strip() for d in os.environ["PII_BENCH_NOTEBOOK_DETECTORS"].split(",") if d.strip()]
if any(d.startswith("presidio") for d in DETECTORS):
    nb.ensure_spacy_model("en_core_web_lg")
if any(d.startswith("skyflow") for d in DETECTORS):
    found = nb.load_secrets(*nb.SKYFLOW_SECRETS)   # from the shell or .env or Colab Secrets
    missing = [k for k, ok in found.items() if not ok]
    if missing:
        raise RuntimeError(f"Skyflow needs {' and '.join(missing)}. Add them under Colab Secrets "
                           "or to .env, or remove Skyflow from DETECTORS.")
    print("Skyflow credentials: loaded")

S = nb.Session(fixtures=str(fx), detectors=DETECTORS)   # not saved on purpose
S.show()
run_dir = nb.ensure_run(S)

### Spans in context

Every span is put back on its whole document together with its page or cell or line. Read a few documents closely because detector disagreements show up here before any estimate does.

In [ ]:
spans = {d: fixtures.map_spans_to_documents(fx, run_dir, d) for d in DETECTORS}
nb.show_table([
    {"detector": d, "spans": len(rows), "documents with spans": len({r["doc_id"] for r in rows}),
     "labels": ", ".join(f"{k} {v}" for k, v in Counter(r["label"] for r in rows).most_common())}
    for d, rows in spans.items()
])
all_spans = [r for rows in spans.values() for r in rows]
for doc in shown_docs(docs):
    nb.show_md(f"### {doc_name(doc)}")
    nb.show_spans_in_context(doc.text, all_spans, doc_id=doc.id, width=40, limit=25)

### The normal report

The normal report refuses to score these fixtures because they have no gold spans. It still counts what each detector found. Sections 4 to 6 of this notebook estimate the scores in three other ways.

In [ ]:
nb.show_md(report.build_report(run_dir, detectors=DETECTORS))

## 4 · Option A: silver labels

Silver labels are gold labels from another source. Each labeler reads every chunk and lists the personal data it sees as `{value, label, context}` items. The harness finds each value in the text itself and uses the context to pick the right occurrence because LLMs are unreliable at character offsets. Values it cannot find are kept in a dropped list and are never lost silently. A chunk whose answer runs out of tokens is asked again in two halves.

With two or more labelers you choose how their spans are merged:

1. `union` keeps a span that any labeler found.
2. `majority` keeps a span that more than half of the labelers agree with. With two labelers it is the same as `intersection`.
3. `intersection` keeps a span that every labeler agrees with.

Two spans agree when they have the same category and their overlap is at least half of the characters that either covers. Each labeler's agreement is checked against the kept span itself and never through a third span. When kept spans overlap, the one with more agreeing labelers wins.

The normal report scores the detectors against the silver file because it is an ordinary fixtures file. Those scores measure agreement with the LLMs and not ground truth, and the report says so under its title. Agreement between two labelers is span F1 with exact boundaries and the same category. Low agreement means the silver labels are noisy.

The defaults are Claude Opus 5.5 through the Anthropic API and GPT-6.1 Sol through the OpenAI API. The commented Vertex AI options keep the text inside your own GCP project. The commented `openai_compatible` option keeps it on this machine.

In [ ]:
# === EDIT ME ===
RUN_SILVER = True
LABELERS = [                          # (provider, model) or (provider, model, base_url)
    ("anthropic", "claude-opus-5-5"),
    ("openai", "gpt-6.1-sol"),        # also gpt-6-luna, gpt-5.6-sol and gpt-5.6-luna
    # Vertex AI in your own GCP project (needs GOOGLE_CLOUD_PROJECT):
    # ("anthropic_vertex", "claude-opus-5-5"),
    # ("openai_vertex", "openai/gpt-oss-120b-maas"),
    # A local OpenAI-compatible server such as Ollama or vLLM:
    # ("openai_compatible", "qwen3:8b", "http://localhost:11434/v1"),
]
MERGE = "union"   # "union" | "majority" | "intersection"
# ================
SILVER_PATH = FIXTURES.with_name(f"{FIXTURES.stem}.silver.jsonl")
TOY_KINDS = [list(TOY_PATTERNS), ["EMAIL", "PHONE"]]   # dry run: the second toy labeler finds less

silver_labelers, silver_path = [], None
if not RUN_SILVER:
    print("skipped: RUN_SILVER is False")
else:
    for k, spec in enumerate(LABELERS):
        client, why = llm_client(*spec, stub_responder=toy_labeler(TOY_KINDS[k % 2]))
        if client is None:
            print(f"labeler {' '.join(spec[:2])} skipped: {why}")
        else:
            silver_labelers.append(client)
            print(f"labeler: {llm.describe(client)}")
    if not silver_labelers:
        print("skipped: no labeler can run here")

In [ ]:
if silver_labelers:
    plan = silver.plan_calls(fx, silver_labelers)
    prompt_chars = len(silver.build_prompt("", read_meta(fx)["labels"])[0])
    tokens_in = (plan["n_calls"] * prompt_chars + plan["n_chars"] * plan["n_labelers"]) // 4
    # An answer is rarely longer than the text because it holds only each value with its label and a little context.
    tokens_out = plan["n_chars"] * plan["n_labelers"] // 4 + plan["n_calls"] * 50
    print(f"{plan['n_labelers']} labeler(s) × {plan['n_records']} chunks of {plan['n_chars']:,} characters in total.")
    print(call_estimate(plan["n_calls"], tokens_in, tokens_out))
    silver_path = silver.generate(fx, silver_labelers, SILVER_PATH, merge_how=MERGE)
    silver_meta = read_meta(silver_path)
    for name, n_err in silver_meta["n_errors"].items():
        if n_err:
            rows = read_jsonl(silver_path.parent / silver_meta["labeler_files"][name])
            first = next(r["error"] for r in rows if r["error"])
            print(f"labeler {name} failed on {n_err} of {len(rows)} chunks. The first error was: {first}")
    if silver_meta["n_written"] and len(silver_meta["error_ids"]) == silver_meta["n_written"]:
        print("The silver labels are not used because every labeler failed on every chunk. "
              "Check the keys and the quota and the errors printed here.")
        silver_path = None
else:
    print("skipped: no labeler can run here")

### Agreement between labelers

The first table compares every pair of labelers. Each row treats `a` as the reference. Precision is therefore the share of `b`'s spans that `a` also found. The second table counts each labeler's failed chunks and the values it named that could not be found in the text.

In [ ]:
if silver_path:
    silver_meta = read_meta(silver_path)
    if silver_meta["agreement"]:
        nb.show_table(silver_meta["agreement"],
                      ["a", "b", "f1", "precision", "recall", "n_a", "n_b", "matched", "n_records"],
                      headers={"f1": "span F1", "n_a": "spans of a", "n_b": "spans of b",
                               "n_records": "records compared"})
    else:
        print("agreement needs two or more labelers")
    nb.show_table([{"labeler": name, "errors": silver_meta["n_errors"][name],
                    "dropped values": silver_meta["n_dropped"][name],
                    "raw answers": silver_meta["labeler_files"][name]}
                   for name in silver_meta["gold_source"]["names"]])
else:
    print("skipped: no silver labels")

### The silver-label report

This is the same report as in [03](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/03_score_and_compare.ipynb) with the silver labels standing in for gold.

In [ ]:
if silver_path:
    nb.show_md(report.build_report(run_dir, fixtures=silver_path, detectors=DETECTORS))
else:
    print("skipped: no silver labels")

## 5 · Option B: LLM review

An LLM reviewer reads each chunk together with every span any detector found in it. A span that several detectors found is judged once and the verdict counts for each of them. The reviewer gives every span one of four verdicts:

1. `correct` means the span is personal data with the right boundaries and category.
2. `wrong_label` means the span is personal data of another category.
3. `boundary` means the category is right but the span is too long or too short.
4. `not_pii` means the span is not personal data.

The reviewer also lists personal data that no detector marked. Estimated precision is the share of a detector's spans judged `correct`, and the lenient view also counts `boundary`. Estimated recall divides the entities a detector found by every entity the review knows of. Those entities are the spans judged `correct` or `wrong_label` or `boundary` plus the reviewer's misses. Overlapping spans judged `correct` count as one entity, and any other accepted span joins the one entity it overlaps most. This is the pooling method from information retrieval evaluation.

**Estimated recall is an upper bound.** PII that every detector and the reviewer overlooked is invisible to it.

In [ ]:
# === EDIT ME ===
RUN_LLM_REVIEW = True
REVIEWER = ("anthropic", "claude-opus-5-5")   # or ("openai", "gpt-6.1-sol") or ("anthropic_vertex", "claude-opus-5-5")
# A local server: ("openai_compatible", "qwen3:8b", "http://localhost:11434/v1")
LENIENT = False                               # True: a boundary verdict also counts as correct
# ================
reviewer, review_path = None, None
if not RUN_LLM_REVIEW:
    print("skipped: RUN_LLM_REVIEW is False")
else:
    reviewer, why = llm_client(*REVIEWER, stub_responder=toy_reviewer)
    if reviewer is None:
        print(f"skipped: the reviewer {REVIEWER[0]} {REVIEWER[1]} cannot run here: {why}")
    else:
        print(f"reviewer: {llm.describe(reviewer)}")
        est = review_llm.estimate(run_dir, detectors=DETECTORS)
        prompt_chars = len(review_llm.build_prompt("", [], read_meta(fx)["labels"])[0])
        tokens_in = (est["n_calls"] * prompt_chars + est["n_chars"] + est["n_candidates"] * 150) // 4
        # A verdict per candidate plus the misses, which are rarely longer than the text.
        tokens_out = est["n_candidates"] * 40 + est["n_chars"] // 8 + est["n_calls"] * 50
        print(f"The reviewer judges {est['n_candidates']} candidate spans in {est['n_chars']:,} characters.")
        print(call_estimate(est["n_calls"], tokens_in, tokens_out))
        review_path = review_llm.review_run(run_dir, reviewer, detectors=DETECTORS)
        review_meta = json.loads(review_llm.review_meta_path(review_path).read_text())
        if review_meta["n_errors"]:
            first = next(r["error"] for r in read_jsonl(review_path) if r["kind"] == "error")
            print(f"The reviewer failed on {review_meta['n_errors']} of {review_meta['n_records']} chunks "
                  f"and those chunks are left out of every estimate. The first error was: {first}")
        if review_meta["n_records"] and review_meta["n_errors"] == review_meta["n_records"]:
            print("The LLM review is not used because the reviewer failed on every chunk. "
                  "Check the key and the quota and the error printed here.")
            review_path = None

In [ ]:
if review_path:
    review_rows = review_llm.summarize(run_dir, review_path, lenient=LENIENT)
    nb.show_table(review_rows, ["detector", "n_spans", "correct", "wrong_label", "boundary", "not_pii",
                                "precision", "recall", "f1", "missed_by_all"],
                  headers={"n_spans": "judged spans", "recall": "recall (upper bound)",
                           "missed_by_all": "missed by all"})
    print("Estimated recall is an upper bound: PII that every detector and the reviewer overlooked is invisible.")
else:
    print("skipped: no LLM review")

### Disagreements in context

These are the spans the reviewer did not accept as correct and the values every detector missed. Each one is shown on its document with its page or cell or line. The label column gives the verdict and then the span's label in parentheses.

In [ ]:
if review_path:
    dis = review_llm.disagreements(run_dir, review_path, limit=None)
    print(f"{len(dis)} disagreement(s): spans not judged correct and values every detector missed")
    by_doc = {d.id: d for d in docs}
    shown = []
    for r in dis:
        doc = by_doc.get(r.get("doc_id"))
        if doc is None:
            continue
        verdict = r["verdict"] + (f" → {r['corrected_label']}" if r.get("corrected_label") else "")
        shown.append({"doc_id": doc.id, "doc_start": r["doc_start"], "doc_end": r["doc_end"],
                      "label": verdict, "fine_label": r["label"],
                      "detector": ", ".join(r["detectors"]) or "none",
                      "where": doc.locate(r["doc_start"], r["doc_end"])})
    for doc in shown_docs(d for d in docs if any(s["doc_id"] == d.id for s in shown)):
        nb.show_md(f"### {doc_name(doc)}")
        nb.show_spans_in_context(doc.text, [s for s in shown if s["doc_id"] == doc.id],
                                 doc_id=doc.id, width=40, limit=25)
else:
    print("skipped: no LLM review")

## 6 · Option C: Clef-flash review

Clef-flash is a typed decision model from Cloudflare. It answers yes/no and choice questions with a probability for each option instead of free text. It runs locally so no text leaves the runtime. It is a cheaper second opinion next to the LLM reviewer. It answers two kinds of question:

1. **Per span.** Is the marked text personal data, and is the proposed category right? A span counts as correct when both probabilities reach the threshold. The share of a detector's spans that count as correct is its estimated precision.
2. **Per sentence.** Each sentence is shown with one detector's spans replaced by placeholders such as `[EMAIL]`, and the model says whether personal data is still left. The share of sentences where it says yes is that detector's **residual-PII rate**. Lower is better.

The cell checks the runtime's GPU before it loads anything. An L4 loads Clef-flash as is because its bf16 weights need about 18.8 GB. A T4 can try 8-bit weights, but that path has not been tested yet and its probabilities may differ from an L4 run. A card with enough memory but no native bf16, such as a V100, loads float16 weights. Float16 can overflow on a model trained in bf16, and a request that overflows becomes an error instead of a number. The cell explains the options when the runtime is too small. The first run downloads about 19 GB of weights from Hugging Face. Call `backend.close()` to free the GPU memory once you are done with this section and the calibration.

The cell prints how many requests failed. Failed requests are left out of every estimate. When every request fails, for example after an out-of-memory error, the section's tables are skipped instead of showing empty numbers.

The tables below use `CLEF_THRESHOLD` as it is when they run. You can change it and rerun only the table cells without reviewing again.

In [ ]:
# === EDIT ME ===
RUN_CLEF = True
CLEF_THRESHOLD = 0.5   # the probability at which an answer counts as yes
# ================
gpu = nb.gpu_summary()
memory = f"{gpu['vram_gb']} GiB" if gpu["vram_gb"] is not None else "unknown memory"
print(f"accelerator: {gpu['device']} ({gpu['name']}), {memory}, native bf16: {gpu['bf16']}")
plan = clef.plan_load()
print(f"Clef-flash load plan: {plan['mode']} because {plan['reason']}")


def clef_failed(path):
    """Print how many Clef-flash requests failed. True when every request failed."""
    meta = json.loads(classifier.review_meta_path(path).read_text())
    n_requests = meta["n_span_requests"] + meta["n_segment_requests"]
    if meta["n_errors"]:
        first = next(r["error"] for r in read_jsonl(path) if r["kind"] == "error")
        print(f"Clef-flash failed on {meta['n_errors']} of {n_requests} requests and those are left out "
              f"of every estimate. The first error was: {first}")
    else:
        print(f"Clef-flash answered all {n_requests} requests")
    return n_requests > 0 and meta["n_errors"] == n_requests


backend, clef_path = None, None
if not RUN_CLEF:
    print("skipped: RUN_CLEF is False")
elif USE_STUB:
    backend = review.StubBackend(toy_decisions)
    print("dry run: a stub backend stands in for Clef-flash")
elif plan["mode"] not in clef.MODES:
    print(f"skipped: Clef-flash cannot load here because {plan['reason']}")
    print("On Colab choose Runtime > Change runtime type > L4 GPU. A T4 can try 8-bit weights, but that path "
          "is untested and its probabilities may differ from an L4 run. TPU runtimes are not supported.")
else:
    if plan["mode"] == "int8":
        print("warning: the 8-bit path is untested and its probabilities may differ from an L4 run. "
              "Treat these estimates as a rough signal and use an L4 when you can.")
    elif plan["mode"] == "fp16":
        print("warning: this card has no native bf16, so Clef-flash runs in float16. Float16 can overflow "
              "and a request that overflows is reported as failed. Use an L4 when you can.")
    try:
        backend = clef.ClefBackend()
    except ImportError as e:
        print(f"skipped: {e}")
if backend is not None:
    clef_path = classifier.review_run(run_dir, backend, detectors=DETECTORS, threshold=CLEF_THRESHOLD)
    if clef_failed(clef_path):
        print("The Clef-flash review is not used because every request failed.")
        clef_path = None

In [ ]:
if clef_path:
    clef_rows = classifier.summarize(run_dir, clef_path, threshold=CLEF_THRESHOLD)
    nb.show_table(clef_rows, ["detector", "n_spans", "precision", "precision_any_label", "mean_p_pii",
                              "n_segments", "residual_rate"],
                  headers={"n_spans": "spans", "precision_any_label": "precision (any label)",
                           "mean_p_pii": "mean p(PII)", "n_segments": "sentences",
                           "residual_rate": "residual-PII rate"})
    print("The residual-PII rate is the share of sentences that still hold personal data after the "
          "detector's own spans are masked. It counts sentences and not values. A sentence with two "
          "leftover values counts once. It is a recall signal and not recall itself.")
else:
    print("skipped: no Clef-flash review")

In [ ]:
if clef_path:
    clef_review = read_jsonl(clef_path)
    records = {r["id"]: r for r in read_jsonl(fx)}
    doubtful = sorted((r for r in clef_review if r["kind"] == "span"), key=lambda r: r["p_pii"])[:10]
    nb.show_md("**The spans the model doubts most**")
    nb.show_table([{"span": r["text"], "label": r["label"], "detectors": ", ".join(r["detectors"]),
                    "p(PII)": r["p_pii"], "p(label ok)": r["p_label_ok"], "best label": r["best_label"]}
                   for r in doubtful])
    leftovers = sorted((r for r in clef_review if r["kind"] == "segment" and r["p_residual"] >= CLEF_THRESHOLD),
                       key=lambda r: -r["p_residual"])[:10]
    nb.show_md("**The sentences most likely to hold leftover PII**")
    nb.show_table([{"detector": r["detector"], "p(left over)": r["p_residual"], "type": r["residual_type"],
                    "sentence": records[r["id"]]["text"][r["start"]:r["end"]][:120]}
                   for r in leftovers])
else:
    print("skipped: no Clef-flash review")

## 7 · Compare the estimates

The table puts every estimate that ran next to the others for each detector. The methods measure slightly different things:

1. Silver precision is the share of a detector's distinct spans that match a merged silver span exactly. Silver recall is the share of silver spans that the detector found exactly. Exactly means the same boundaries and the same category.
2. LLM-review precision counts spans judged `correct`. LLM-review recall is an upper bound.
3. Clef-flash precision counts spans where both answers reach the threshold. The residual-PII rate falls as recall rises.

With `LENIENT = False` perfect silver labels and a perfect reviewer give the same numbers because silver and LLM-review scores count in the same way. The silver-label report in section 4 uses nervaluate's strict schema instead. Its recall can be lower than the silver recall here because nervaluate adds every wrong-boundary span that overlaps an already-found silver span to the recall denominator.

Agreement between methods is reassuring. The last column is the spread between the highest and the lowest precision estimate for a detector. A detector with a wide spread deserves a closer look at its spans in context and at the LLM review's disagreements in context.

In [ ]:
def exact_scores(run, ids=None):
    """Exact-match precision and recall per detector, counted the way the LLM review counts.

    Precision is the share of a detector's distinct spans with a gold span of the same
    boundaries and coarse category. Recall is the share of gold spans the detector found
    that way. Gold spans in categories the detector does not support still count, as they
    do in the review. Records the detector errored on are left out.
    """
    out = {}
    for det in DETECTORS:
        if det not in run.detectors:
            continue
        scope = scoring.scope_for(run, det, view="raw")
        n_pred = n_gold = hits = 0
        for pred, gold in scoring.build_pairs(run, det, scope, ids=ids).pairs:
            p = {(s["start"], s["end"], s["label"]) for s in pred}
            g = {(s["start"], s["end"], s["label"]) for s in gold}
            n_pred, n_gold, hits = n_pred + len(p), n_gold + len(g), hits + len(p & g)
        out[det] = {"P": hits / n_pred if n_pred else None, "R": hits / n_gold if n_gold else None}
    return out


compare = {d: {"detector": d} for d in DETECTORS}
if silver_path:
    silver_meta = read_meta(silver_path)
    silver_run = scoring.Run.load(run_dir, silver_path)
    # Score on the labelers' labels and without the chunks that no labeler answered.
    silver_run.manifest = {**silver_run.manifest, "labels": list(silver_meta["labels"])}
    failed = set(silver_meta.get("error_ids") or [])
    scored = exact_scores(silver_run, ids=[r["id"] for r in silver_run.fixtures if r["id"] not in failed])
    for d, s in scored.items():
        compare[d].update({"silver P": s["P"], "silver R": s["R"]})
if review_path:
    for r in review_llm.summarize(run_dir, review_path, lenient=LENIENT):
        if r["detector"] in compare:
            compare[r["detector"]].update({"review P": r["precision"], "review R (upper bound)": r["recall"]})
if clef_path:
    for r in classifier.summarize(run_dir, clef_path, threshold=CLEF_THRESHOLD):
        if r["detector"] in compare:
            compare[r["detector"]].update({"Clef P": r["precision"], "Clef residual rate": r["residual_rate"]})

compare_rows = list(compare.values())
for row in compare_rows:
    estimates = [row[k] for k in ("silver P", "review P", "Clef P") if row.get(k) is not None]
    row["P spread"] = max(estimates) - min(estimates) if len(estimates) >= 2 else None
nb.show_table(compare_rows, ["detector", "silver P", "silver R", "review P", "review R (upper bound)",
                             "Clef P", "Clef residual rate", "P spread"])
spread = [r for r in compare_rows if r["P spread"] is not None]
if spread:
    worst = max(spread, key=lambda r: r["P spread"])
    print(f"The precision estimates disagree most for {worst['detector']}: "
          f"they differ by {worst['P spread']:.2f}.")
else:
    print("Run at least two of options A to C to compare their precision estimates.")
if USE_STUB:
    print("dry run: these numbers come from toy stand-ins and mean nothing")

## 8 · Calibrate on a gold sample

Check each method where the truth is known before you trust it on your data. This section reruns each method that ran on your data on a sample of `pii_masking_200k`, which has real gold spans. It shows each estimate next to the true score:

1. **Silver labels.** `silver.calibrate` labels the sample with each labeler and scores the silver spans against the real gold. A labeler with a low F1 here also writes noisy silver labels for your data.
2. **LLM review.** The reviewer judges the detectors' spans on the sample. Its precision and recall estimates appear next to each detector's true precision and recall.
3. **Clef-flash.** `classifier.threshold_sweep` compares estimated precision and residual-PII rates with the truth at several thresholds. Pick the threshold where the error columns are closest to zero and set `CLEF_THRESHOLD` to it.

The true precision and recall count exact matches with the gold spans in the same way as the silver scores in section 7. The Clef-flash table has its own true precision column, which counts only the spans the classifier reviewed. Those are the detector's distinct spans in the categories the review targets. Both true precision columns normally agree on this sample because those categories are the ones the sample annotates.

The section is off by default because it costs another round of model calls. The detectors need no GPU here because the default sample of 100 records matches the baseline that ships with the harness. The first run downloads the dataset sample from Hugging Face.

In [ ]:
# === EDIT ME ===
RUN_CALIBRATION = False
CALIBRATION_DATASET = "pii_masking_200k"
CALIBRATION_N = 100
# ================
calibrating = False
if not RUN_CALIBRATION:
    print("skipped: RUN_CALIBRATION is False")
elif not (silver_labelers or reviewer or backend):
    print("skipped: there is nothing to calibrate because options A to C were all skipped")
else:
    calibrating = True
    gold_session = nb.Session(dataset=CALIBRATION_DATASET, n=CALIBRATION_N, detectors=DETECTORS,
                              run_name=f"calibration_{CALIBRATION_DATASET}_{CALIBRATION_N}")
    gold_run = nb.ensure_run(gold_session)
    gold_fx = gold_session.fixtures_path
    true_scores = {d: {"true P": s["P"], "true R": s["R"]}
                   for d, s in exact_scores(scoring.Run.load(gold_run)).items()}
    nb.show_md("**True exact-match scores on the gold sample**")
    nb.show_table([{"detector": d, **s} for d, s in true_scores.items()])

In [ ]:
if calibrating and silver_labelers:
    # calibrate labels the same records as the gold sample above (same dataset, size and seed).
    plan = silver.plan_calls(gold_fx, 1)
    prompt_chars = len(silver.build_prompt("", read_meta(gold_fx)["labels"])[0])
    rows = []
    for labeler in silver_labelers:
        if labeler.remote and not nb.allow_remote():
            print(f"labeler {llm.describe(labeler)} skipped: it would send the text to {labeler.operator} "
                  "and hosted LLMs are switched off")
            continue
        print(f"labeler: {llm.describe(labeler)}")
        print(call_estimate(plan["n_calls"], (plan["n_calls"] * prompt_chars + plan["n_chars"]) // 4,
                            plan["n_chars"] // 4 + plan["n_calls"] * 50))
        cal = silver.calibrate(labeler, dataset=CALIBRATION_DATASET, n=CALIBRATION_N,
                               workdir=ws.data / "calibration")
        rows.append({"labeler": cal["labeler"], "strict P": cal["strict"]["precision"],
                     "strict R": cal["strict"]["recall"], "strict F1": cal["strict"]["f1"],
                     "type F1": cal["ent_type"]["f1"], "records": cal["n"], "errors": cal["n_errors"]})
    if rows:
        nb.show_md("**Silver labelers against real gold**")
        nb.show_table(rows)
elif calibrating:
    print("silver labels: skipped because option A did not run")

In [ ]:
if calibrating and reviewer:
    est = review_llm.estimate(gold_run, detectors=DETECTORS)
    print(f"The reviewer judges {est['n_candidates']} candidate spans on the gold sample.")
    print(call_estimate(est["n_calls"], (est["n_calls"] * 2000 + est["n_chars"] + est["n_candidates"] * 150) // 4,
                        est["n_candidates"] * 40 + est["n_chars"] // 8 + est["n_calls"] * 50))
    gold_review = review_llm.review_run(gold_run, reviewer, detectors=DETECTORS)
    gold_meta = json.loads(review_llm.review_meta_path(gold_review).read_text())
    if gold_meta["n_errors"]:
        print(f"The reviewer failed on {gold_meta['n_errors']} of {gold_meta['n_records']} records "
              "and those records are left out of the estimates.")
    nb.show_md("**LLM-review estimates against the true scores**")
    nb.show_table([{"detector": r["detector"], "estimated P": r["precision"],
                    "true P": true_scores.get(r["detector"], {}).get("true P"),
                    "estimated R (upper bound)": r["recall"],
                    "true R": true_scores.get(r["detector"], {}).get("true R")}
                   for r in review_llm.summarize(gold_run, gold_review, lenient=LENIENT)])
elif calibrating:
    print("LLM review: skipped because option B did not run")

In [ ]:
gold_clef = None
if not calibrating:
    pass
elif backend is None:
    print("Clef-flash: skipped because option C did not run")
elif getattr(backend, "model", True) is None:
    print("Clef-flash: skipped because the backend was closed. Rerun section 6 to load it again.")
else:
    gold_clef = classifier.review_run(gold_run, backend, detectors=DETECTORS, threshold=CLEF_THRESHOLD)
    if clef_failed(gold_clef):
        print("Clef-flash: the threshold table is skipped because every request failed.")
        gold_clef = None
if gold_clef:
    nb.show_md("**Clef-flash estimates against the truth at each threshold**")
    nb.show_table(classifier.threshold_sweep(gold_run, gold_clef, gold_fx),
                  ["threshold", "detector", "estimated_precision", "true_precision", "precision_error",
                   "estimated_residual_rate", "true_residual_rate", "residual_error"],
                  headers={"estimated_precision": "est. P", "true_precision": "true P (reviewed spans)",
                           "precision_error": "P error", "estimated_residual_rate": "est. residual",
                           "true_residual_rate": "true residual", "residual_error": "residual error"})
    print("An error above zero means the classifier is too optimistic about precision "
          "or too pessimistic about leftover PII.")

## Where next

1. Choose a detector or a composite from [04](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/04_composite_detectors.ipynb) with the estimate that held up best in calibration.
2. Move the LLM calls to the Vertex AI providers before a wider rollout so the text stays in your own GCP project.
3. Keep these workspace folders private because they hold your data or text quoted from it. Inside a git checkout the notebook gives each of them a `.gitignore` that ignores everything in it.
    1. `inputs` holds your files.
    2. `llm_cache` holds the models' answers.
    3. `data/custom` holds the fixtures and silver labels.
    4. `data/calibration` holds the calibration samples.
    5. `results/runs` holds the detector results and the review files.
4. Keep the outputs of this notebook out of git because they quote your data.